# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. output: a=[1,2,3], a=[0,1,1], потому что это массив. Опирация идет внутри
2. output: m[[4],[4],[4]] axis схлопывает по строкам и и берем sum этой схлоповшейся строки 
3. output: [[0 1 2]
    [1 2 3]  3 на 3 матрикс np.arange(3) создает [0,1,2] и np.arange(3).reshape(3, 1) создает [0,1,2] но как столбец и между собой складыветься
    [2 3 4]]
4. output: 6    корень 1,4,9 этих цифр и складываем
5. output: 6.5    [1,4,9,16] mean = 9+4/2 = 6.5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [30]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [31]:
# 1
print(f"Вот сколько строк осталось в clean: {int(clean.sum())}")
rev_sept = amount[(clean) & (month == "2026-09")].sum()
print(f"Выручки в сентябре: {rev_sept:.2f}")
rev_oct = amount[(clean) & (month == "2026-10")].sum()
print(f"Выручки в октябре: {rev_oct:.2f}")
percent_change = ((rev_oct - rev_sept) / rev_sept) * 100
print(f"Процент изменения выручки: {percent_change:.2f}%")


Вот сколько строк осталось в clean: 4739
Выручки в сентябре: 35243410.00
Выручки в октябре: 34419520.00
Процент изменения выручки: -2.34%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [40]:

cats = np.unique(cat)
months = np.unique(month)

table = np.array([
    [amount[clean & (cat == c) & (month == m)].sum() for m in months]
    for c in cats
])

print(f"Категория × месяц ({months[0]} | {months[1]}):")
for i, c in enumerate(cats):
    print(f"• {c}: {table[i, 0]:,.0f} тг | {table[i, 1]:,.0f} тг")

total_by_cat = table.sum(axis=1)
best_idx = np.argmax(total_by_cat)
print(f"\nБольше всего выручки за 2 месяца: {cats[best_idx]} ({total_by_cat[best_idx]:,.0f} тг)")

change_pct = (table[:, 1] - table[:, 0]) / table[:, 0] * 100
worst_idx = np.argmin(change_pct)
print(f"Сильнее всего упала: {cats[worst_idx]} ({change_pct[worst_idx]:.2f}%)")

Категория × месяц (2026-09 | 2026-10):
• Дом: 4,997,730 тг | 5,125,790 тг
• Книги: 1,065,300 тг | 1,140,910 тг
• Красота: 2,610,950 тг | 2,678,990 тг
• Одежда: 9,167,100 тг | 8,505,730 тг
• Электроника: 17,402,330 тг | 16,968,100 тг

Больше всего выручки за 2 месяца: Электроника (34,370,430 тг)
Сильнее всего упала: Одежда (-7.21%)


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [46]:
# 3
counts = np.array([
    [np.sum(clean & (cat == c) & (month == m)) for m in months]
    for c in cats
])
avg_check = table / counts
oct_avg = avg_check[:, 1]
print(f"Средний чек по категориям в октябре ({months[1]}):")
for i, c in enumerate(cats):
    print(f"• {c}: {avg_check[i, 1]:,.0f} тг (заказов: {counts[i, 1]:.0f} шт.)")

best_oct_idx = np.argmax(avg_check[:, 1])
print(f"\nСамый высокий средний чек в октябре: {cats[best_oct_idx]} ({avg_check[best_oct_idx, 1]:,.0f} тг)")

Средний чек по категориям в октябре (2026-10):
• Дом: 10,768 тг (заказов: 476 шт.)
• Книги: 3,816 тг (заказов: 299 шт.)
• Красота: 6,159 тг (заказов: 435 шт.)
• Одежда: 13,046 тг (заказов: 652 шт.)
• Электроника: 33,600 тг (заказов: 505 шт.)

Самый высокий средний чек в октябре: Электроника (33,600 тг)


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [34]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [35]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [36]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [37]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [38]:
# 8